<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/local_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1 Mount Google
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/jarwo_tts")

DATASET_DIR = ROOT / "video_dataset"
WAV_DIR = DATASET_DIR / "wavs"
METADATA = DATASET_DIR / "metadata.csv"

EXPORT_DIR = ROOT / "export"
BACKUP_DIR = ROOT / "checkpoints"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

REF_DIR = ROOT / "video_dataset/wavs"
GENERATED = ROOT / "export/test_voice.wav"

refs = sorted(REF_DIR.glob("*.wav"))

print("Reference clips:", len(refs))
print("Generated exists:", GENERATED.exists())

print("Metadata exists :", METADATA.exists())
print("WAV count       :", len(list(WAV_DIR.glob("*.wav"))))

Mounted at /content/drive
Metadata exists : True
WAV count       : 17


In [2]:
# 2. INSTALL FFMPEG WHISPER
%cd /content
!pip install -q transformers librosa scikit-learn
!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    cmake \
    ninja-build \
    git \
    espeak-ng

!rm -rf piper1-gpl

!git clone --depth 1 \
    https://github.com/OHF-Voice/piper1-gpl.git

%cd /content/piper1-gpl

!pip install -q -e ".[train]"

/content
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../0-libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../1-libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../2-espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../3-libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ..

In [3]:
!chmod +x build_monotonic_align.sh
!./build_monotonic_align.sh

!python3 setup.py build_ext --inplace

Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.13/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:38:6: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
Possible solutions:
	

In [4]:
# 3 Check GPU

import torch

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU tidak aktif")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : Tesla T4


In [5]:
# 4 — Check Audio

print(open(METADATA, encoding="utf-8").read())

000001.wav|Di tempat kulahir dan dibesarkan, di kota yang dekat laut ini, setelah
000002.wav|segi nama aku pulang, sudah ada shopping mall berdiri.
000003.wav|Waktu itu selalu seperti tongkacir, walaupun telah mengubah pemandangan suara
000004.wav|ombak dan aroma kewombang masih sama seperti dulu.
000005.wav|Saya berhenti sekarang juga, kamu yang terlis di mewah, ada di pojok
000006.wav|renang bukutahunan kita, sumbuk memang kamu yang terlis di mewah.
000007.wav|Berapa kali aku buka untuk memastikannya?
000008.wav|Kamu berdiri di kasir counter, kita cita kamu menjadi seorang hastylist.
000009.wav|Waktu itu kamu pernah bercerita, walaupun seperti yang kamu bayangkan kamu terlihat
000010.wav|Aku jadi legaku dengan gawau kamu sudah menikah, aku terlambat
000011.wav|bilang suka kepadamu, aku dengar kamu pun sekarang punya anak.
000012.wav|Tak sambung memanggilmu fredor masamudahku.
000013.wav|Sekarang juga kamu yang terlis di mewah, ada di pojok renang bukutahunan kita,
000014.wav|sumbuk m

In [6]:
!pip install -q huggingface_hub

In [7]:
# 5. Download Pretrained
from huggingface_hub import hf_hub_download
from pathlib import Path

BASE_DIR = Path("/content/base_piper")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BASE_CKPT = hf_hub_download(
    repo_id="rhasspy/piper-checkpoints",
    repo_type="dataset",
    filename=(
        "id/id_ID/news_tts/medium/"
        "epoch=4927-step=232092.ckpt"
    ),
    local_dir=str(BASE_DIR)
)

print(BASE_CKPT)

id/id_ID/news_tts/medium/epoch=4927-step(…): reconstructing file:   0%|          |  0.00B /  846MB            

id/id_ID/news_tts/medium/epoch=4927-step(…): downloading bytes:           |  0.00B            

/content/base_piper/id/id_ID/news_tts/medium/epoch=4927-step=232092.ckpt


In [8]:
# 6. Copy dataset Drive → storage Colab
import shutil
from pathlib import Path

LOCAL_DATASET = Path("/content/jarwo_dataset")
LOCAL_WAV = LOCAL_DATASET / "wavs"
LOCAL_METADATA = LOCAL_DATASET / "metadata.csv"

if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)

LOCAL_WAV.mkdir(parents=True, exist_ok=True)

for wav in WAV_DIR.glob("*.wav"):
    shutil.copy2(
        wav,
        LOCAL_WAV / wav.name
    )

shutil.copy2(
    METADATA,
    LOCAL_METADATA
)

print("WAV copied:", len(list(LOCAL_WAV.glob("*.wav"))))
print("Metadata:", LOCAL_METADATA)

WAV copied: 17
Metadata: /content/jarwo_dataset/metadata.csv


In [9]:
# 7. Tentukan lokasi config dan training

CONFIG_FILE = ROOT / "jarwo_voice.json"

RUN_DIR = Path("/content/jarwo_training")
CACHE_DIR = Path("/content/jarwo_cache")

print("Config:", CONFIG_FILE)

Config: /content/drive/MyDrive/jarwo_tts/jarwo_voice.json


In [10]:
# 8. TRAINING
import subprocess
import shutil

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)

RUN_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "python3",
    "-m",
    "piper.train",
    "fit",

    "--data.voice_name",
    "jarwo_voice",

    "--data.csv_path",
    str(LOCAL_METADATA),

    "--data.audio_dir",
    str(LOCAL_WAV),

    "--model.sample_rate",
    "22050",

    "--data.espeak_voice",
    "id",

    "--data.cache_dir",
    str(CACHE_DIR),

    "--data.config_path",
    str(CONFIG_FILE),

    "--data.batch_size",
    "2",

    "--data.validation_split",
    "0.10",

    "--data.num_test_examples",
    "1",

    "--data.num_workers",
    "2",

    "--model.warmstart_ckpt",
    str(BASE_CKPT),

    "--model.mos_metric",
    "none",

    "--trainer.accelerator",
    "gpu",

    "--trainer.devices",
    "1",

    "--trainer.default_root_dir",
    str(RUN_DIR),

    "--trainer.max_steps",
    "500",

    "--trainer.max_epochs",
    "500",

    "--trainer.log_every_n_steps",
    "5",
]

print("=== START TRAINING ===")

subprocess.run(
    cmd,
    cwd="/content/piper1-gpl",
    check=True
)

=== START TRAINING ===


CalledProcessError: Command '['python3', '-m', 'piper.train', 'fit', '--data.voice_name', 'jarwo_voice', '--data.csv_path', '/content/jarwo_dataset/metadata.csv', '--data.audio_dir', '/content/jarwo_dataset/wavs', '--model.sample_rate', '22050', '--data.espeak_voice', 'id', '--data.cache_dir', '/content/jarwo_cache', '--data.config_path', '/content/drive/MyDrive/jarwo_tts/jarwo_voice.json', '--data.batch_size', '2', '--data.validation_split', '0.10', '--data.num_test_examples', '1', '--data.num_workers', '2', '--model.warmstart_ckpt', '/content/base_piper/id/id_ID/news_tts/medium/epoch=4927-step=232092.ckpt', '--model.mos_metric', 'none', '--trainer.accelerator', 'gpu', '--trainer.devices', '1', '--trainer.default_root_dir', '/content/jarwo_training', '--trainer.max_steps', '500', '--trainer.max_epochs', '500', '--trainer.log_every_n_steps', '5']' returned non-zero exit status 1.

In [ ]:
# 9. Cari checkpoint
from pathlib import Path

ckpts = list(
    RUN_DIR.rglob("*.ckpt")
)

print("Checkpoint ditemukan:", len(ckpts))

for ckpt in ckpts:
    print(ckpt)

In [ ]:
# 10. Backup checkpoint ke Drive
import shutil

DRIVE_CKPT = (
    CHECKPOINT_DIR /
    "jarwo_500steps.ckpt"
)

shutil.copy2(
    FINAL_CKPT,
    DRIVE_CKPT
)

print("Backup ✅")
print(DRIVE_CKPT)

In [ ]:
# 11. Export menjadi ONNX
ONNX_FILE = (
    EXPORT_DIR /
    "jarwo_test.onnx"
)

subprocess.run(
    [
        "python3",
        "-m",
        "piper.train.export_onnx",

        "--checkpoint",
        str(FINAL_CKPT),

        "--output-file",
        str(ONNX_FILE),
    ],
    cwd="/content/piper1-gpl",
    check=True
)

print("ONNX:")
print(ONNX_FILE)

In [ ]:
# 12. Copy config
from pathlib import Path
import shutil

ONNX_CONFIG = Path(
    str(ONNX_FILE) + ".json"
)

shutil.copy2(
    CONFIG_FILE,
    ONNX_CONFIG
)

print(ONNX_FILE)
print(ONNX_CONFIG)

In [ ]:
# 13. Test suara
TEST_TEXT = (
    "Halo, sekarang saya sedang mencoba suara baru. "
    "Baterai perangkat masih enam puluh lima persen."
)

TEST_WAV = (
    EXPORT_DIR /
    "test_voice.wav"
)

In [ ]:
result = subprocess.run(
    [
        "python3",
        "-m",
        "piper",

        "-m",
        str(ONNX_FILE),

        "-c",
        str(ONNX_CONFIG),

        "-f",
        str(TEST_WAV),

        "--",
        TEST_TEXT,
    ],
    cwd="/content/piper1-gpl",
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("TTS gagal")

print(TEST_WAV)

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        str(TEST_WAV)
    )
)

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

REFERENCE = refs[0]

def load_audio(path):
    y, sr = librosa.load(path, sr=22050, mono=True)
    return y, sr


ref_audio, sr = load_audio(REFERENCE)
gen_audio, _ = load_audio(GENERATED)


plt.figure(figsize=(12, 3))

librosa.display.waveshow(
    ref_audio,
    sr=sr
)

plt.title("Reference Voice")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
plt.figure(figsize=(12, 3))

librosa.display.waveshow(
    gen_audio,
    sr=sr
)

plt.title("Fine-tuned TTS")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
mel_ref = librosa.feature.melspectrogram(
    y=ref_audio,
    sr=sr,
    n_mels=80
)

mel_ref_db = librosa.power_to_db(
    mel_ref,
    ref=np.max
)

plt.figure(figsize=(12, 4))

librosa.display.specshow(
    mel_ref_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Mel Spectrogram — Reference")
plt.show()

In [ ]:
mel_gen = librosa.feature.melspectrogram(
    y=gen_audio,
    sr=sr,
    n_mels=80
)

mel_gen_db = librosa.power_to_db(
    mel_gen,
    ref=np.max
)

plt.figure(figsize=(12, 4))

librosa.display.specshow(
    mel_gen_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Mel Spectrogram — Fine-tuned TTS")
plt.show()

In [ ]:
import torch

from transformers import (
    AutoFeatureExtractor,
    WavLMForXVector
)

MODEL_NAME = "microsoft/wavlm-base-plus-sv"

feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_NAME
)

speaker_model = WavLMForXVector.from_pretrained(
    MODEL_NAME
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

speaker_model = speaker_model.to(device)
speaker_model.eval()

print("Speaker model ready:", device)

In [ ]:
import librosa
import torch
import numpy as np

def speaker_embedding(path):

    audio, _ = librosa.load(
        path,
        sr=16000,
        mono=True
    )

    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():

        output = speaker_model(
            input_values
        )

        emb = output.embeddings

    emb = torch.nn.functional.normalize(
        emb,
        dim=-1
    )

    return emb[0].cpu().numpy()

In [ ]:
reference_embeddings = []

for wav in refs:

    emb = speaker_embedding(wav)

    reference_embeddings.append(
        emb
    )

reference_embeddings = np.stack(
    reference_embeddings
)

print(reference_embeddings.shape)

In [ ]:
generated_embedding = speaker_embedding(
    GENERATED
)

print(generated_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(
    reference_embeddings,
    generated_embedding.reshape(1, -1)
).flatten()

for wav, score in zip(refs, scores):

    print(
        wav.name,
        "=>",
        round(float(score), 4)
    )

In [ ]:
print(
    "Average similarity:",
    round(
        float(scores.mean()),
        4
    )
)

print(
    "Best similarity:",
    round(
        float(scores.max()),
        4
    )
)

print(
    "Worst similarity:",
    round(
        float(scores.min()),
        4
    )
)

In [ ]:
plt.figure(
    figsize=(12, 5)
)

plt.bar(
    range(len(scores)),
    scores
)

plt.axhline(
    scores.mean(),
    linestyle="--",
    label=f"Average = {scores.mean():.3f}"
)

plt.xlabel(
    "Reference Clip"
)

plt.ylabel(
    "Cosine Similarity"
)

plt.title(
    "Custom TTS vs Target Speaker"
)

plt.xticks(
    range(len(scores)),
    [
        wav.stem
        for wav in refs
    ],
    rotation=45
)

plt.ylim(
    min(0, scores.min() - 0.1),
    1
)

plt.legend()

plt.show()

In [ ]:
from sklearn.decomposition import PCA

all_embeddings = np.vstack([
    reference_embeddings,
    generated_embedding.reshape(1, -1)
])

pca = PCA(
    n_components=2
)

points = pca.fit_transform(
    all_embeddings
)

ref_points = points[:-1]
gen_point = points[-1]

In [ ]:
plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    ref_points[:, 0],
    ref_points[:, 1],
    label="Target speaker",
    s=60
)

for i, p in enumerate(ref_points):

    plt.annotate(
        str(i + 1),
        (p[0], p[1]),
        fontsize=8
    )

plt.scatter(
    gen_point[0],
    gen_point[1],
    marker="X",
    s=180,
    label="Fine-tuned TTS"
)

plt.xlabel(
    "PCA 1"
)

plt.ylabel(
    "PCA 2"
)

plt.title(
    "Speaker Embedding Map"
)

plt.legend()

plt.show()